# Download Common Voice 26.0 (Swahili) from Mozilla Data Collective

Downloads dataset `cmqim4c1000tmnr07zq3vwhor` (`common-voice-scripted-speech-26-0-swahil-0228b2f6`,
~22 GB compressed) from the Mozilla Data Collective (MDC) platform using the official
`datacollective` Python SDK.

**Before running, in this notebook's settings:**
1. **Settings -> Internet -> On** (required; MDC is an external API).
2. **Add-ons -> Secrets -> Add a new secret**, named exactly `MDC_API_KEY`, with your
   MDC API key as the value. Generate a key at https://mozilladatacollective.com after
   creating an account and agreeing to this dataset's Terms & Conditions on its MDC page.
   *If you ever pasted this key anywhere outside the Kaggle Secrets vault (chat, a script,
   a committed file), rotate/revoke it on the MDC platform and generate a fresh one first.*

The key is only ever read from the Kaggle Secrets vault at runtime -- it is never
printed, logged, or written to a file by this notebook.

In [ ]:
!pip install -q datacollective

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
os.environ["MDC_API_KEY"] = user_secrets.get_secret("MDC_API_KEY")
print("MDC_API_KEY loaded from Kaggle Secrets.")

## Download the archive

Downloaded to `/kaggle/temp`, Kaggle's larger ephemeral scratch space, rather than
`/kaggle/working` -- `/kaggle/working` is what gets persisted as notebook output and has
a much smaller quota than this ~22 GB archive (plus its extracted contents) needs.
`download_dataset` resumes automatically if this cell is re-run after an interruption.

In [ ]:
from pathlib import Path
from datacollective import download_dataset

DATASET_ID = "cmqim4c1000tmnr07zq3vwhor"
DOWNLOAD_DIR = "/kaggle/temp/mdc_common_voice_sw"
Path(DOWNLOAD_DIR).mkdir(parents=True, exist_ok=True)

archive_path = download_dataset(
    DATASET_ID,
    download_directory=DOWNLOAD_DIR,
    show_progress=True,
)
print("Downloaded archive to:", archive_path)

## Extract

Extracts the archive next to the download. This can take a while and needs enough
free disk for both the compressed archive and its extracted contents at once.

In [ ]:
import tarfile

extract_dir = Path(DOWNLOAD_DIR) / "extracted"
extract_dir.mkdir(exist_ok=True)

with tarfile.open(archive_path) as tar:
    tar.extractall(path=extract_dir)

print("Extracted to:", extract_dir)
for p in sorted(extract_dir.rglob("*"))[:20]:
    print(" ", p)

## Optional: load directly as a pandas DataFrame

`load_dataset` downloads (if needed), extracts, and parses the dataset using MDC's
schema registry, returning a ready-to-use DataFrame. It only works if this dataset
has a registered schema; if not, the manually extracted files above (`validated.tsv`,
`clips/`, etc., in the standard Common Voice layout) are the fallback.

In [ ]:
from datacollective import load_dataset

try:
    df = load_dataset(DATASET_ID, download_directory=DOWNLOAD_DIR, show_progress=True)
    print(df.shape)
    display(df.head())
except RuntimeError as e:
    print("No registered schema for this dataset yet -- using the manual extraction instead:")
    print(e)

## Locate the transcript metadata

Finds the Common Voice metadata TSV (`validated.tsv` or similar), which is what
`scripts/select_subset.py` in the Swahili-Deepfake-dataset pipeline consumes
(columns `path`, `client_id`, `sentence` by default).

In [ ]:
import pandas as pd

tsv_candidates = list(extract_dir.rglob("validated.tsv")) or list(extract_dir.rglob("*.tsv"))
print("Found TSV files:")
for p in tsv_candidates:
    print(" -", p)

if tsv_candidates:
    preview = pd.read_csv(tsv_candidates[0], sep="\t", nrows=5)
    display(preview)
    print("Columns:", list(preview.columns))

## Save a lightweight copy of the metadata as notebook output

Copies just the TSVs (not the 22 GB+ of audio clips) into `/kaggle/working` so they
are saved as this notebook's output and downloadable after the session ends. The
audio itself stays in `/kaggle/temp` (ephemeral) -- either process it within this
same session, or publish it as a new Kaggle Dataset via "Save Version" if your
output quota allows it.

In [ ]:
import shutil

output_dir = Path("/kaggle/working/mdc_metadata")
output_dir.mkdir(exist_ok=True)

for p in tsv_candidates:
    shutil.copy(p, output_dir / p.name)

print("Copied metadata TSVs to:", output_dir)